In [20]:
from langgraph.graph import StateGraph,END
from langchain_ollama import ChatOllama
from langchain.tools import Tool,tool
from langchain_community.tools.tavily_search import TavilySearchResults
import os

In [21]:
from typing import TypedDict,Annotated,Sequence
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages : Annotated[Sequence[BaseMessage],operator.add]

In [22]:
os.environ["TAVILY_API_KEY"] = "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"
tavily_tool = TavilySearchResults(max_results=2)

chat_ollama = ChatOllama(model='llama3.2')

In [23]:
@tool 
def multiply(first_number: int, second_number: int) -> int:
    ''' Your task is to multiply two number'''
    return first_number*second_number

@tool 
def web_search(message: str) -> str:
    ''' Search forthe website reated to provided message '''
    response = tavily_tool.invoke({'query':message})
    return response

In [24]:
tool_list = [multiply,web_search]
tool_model = chat_ollama.bind_tools(tool_list)

In [44]:
def invoke_model(state):
    message = state['messages']
    response = tool_model.invoke(message[0])
    state['messages'].append(response.response_metadata.get('message').get('tool_calls')[0])
    return state

def router(state):
    if len(state['messages'][1]['function']['arguments']):
        return 'tools'
    else:
        return 'end'
    
def generate_response(state):
    function = state['messages'][1]['function']['name']
    val =''
    if function == 'web_search':
        val = input('should we continue with web search:')
        if val == 'y':
            response = web_search.invoke(state['messages'][0])
            state['messages'].append({'response':response})    
        else:
            state['messages'].append({'response':'not allowed'})  
    elif function=='multiply':
        args = state['messages'][1]['function']['arguments']
        val = {}
        for key, value in args.items():
            val[key]=int(value)
    
        response = multiply.invoke(val)
        state['messages'].append({'response':response})
    
    return state

In [45]:
workflow = StateGraph(AgentState)

workflow.add_node('invoke',invoke_model)
workflow.add_node('response',generate_response)

workflow.set_entry_point('invoke')

workflow.add_conditional_edges('invoke',router,{'tools':'response','end':END})

app=workflow.compile()

In [52]:
answer = app.invoke({'messages':['who is narendra modi ?']})
answer['messages'][-1]

{'response': 'not allowed'}

In [53]:
answer = app.invoke({'messages':['what is thevalue of 3*2 ?']})
answer['messages'][-1]

{'response': 6}

In [54]:
answer = app.invoke({'messages':['who is narendra modi ?']})
answer['messages'][-1]

{'response': [{'url': 'https://www.pmindia.gov.in/en/personal_life_story/personal-life-story/',
   'content': 'To read more about Shri Narendra Modi’s personal life story please visit: http://www.narendramodi.in/humble-beginnings-the-early-years/\nYears in Governance\nNarendra Modi’s evolution from quintessential Organization Man of the BJP to one of India’s best known leaders recognized for his Good Governance over a span of a decade tells a story of grit, determination and Strong Leadership in the face of grave adversity. His focus on development, eye for detail and efforts to bring a qualitative difference in the lives of the poorest of the poor have made Narendra Modi a popular and respected leader across the length and breadth of India.\n PMINDIA\nPersonal Life Story\nHistory was scripted in the forecourt of Rashtrapati Bhawan on the evening of 26th May 2014 as Narendra Modi took oath as the Prime Minister of India after a historic mandate from the people of India. Narendra Modi’s